# Q2D Semantic Cross-Section Two-Die Offset Sweep End-to-End Handoff

This notebook creates one two-die semantic Q2D parameter space and writes one AEDT handoff package. D0 top has trace T1, D1 bottom has trace T2, and the flip-chip gap height is an explicit sweep axis.


In [ ]:
from __future__ import annotations

import csv
import json
from datetime import date
from pathlib import Path
from tempfile import TemporaryDirectory

from orpen_sc_pdk.config import PATH
from orpen_sc_pdk.simulation.aedt import (
    AedtHpcResourceSpec,
    AedtNativeCaseSpec,
    AedtNativePackageSpec,
    AedtRecipeSpec,
    Air,
    Axis,
    Die,
    DieGap,
    FacePattern,
    Gap,
    Ground,
    ParameterSpace,
    Q2dFacetLineGrid,
    Q2dImpedanceFormula,
    Q2dSemanticCrossSection,
    Stack,
    Trace,
    load_q2d_raw_sweep_result,
    package_aedt_native_handoff,
    prepare_aedt_native_handoff_package,
    write_q2d_cross_section_payload,
)

In [ ]:
# AEDT run-folder controls. The run folder itself is the handoff package.
# Set NOTEBOOK_RUN_ID to an existing run folder name to expand it in place.
# Package regeneration updates handoff files and preserves existing results/logs/points.
SIMULATION_PURPOSE_ID = "q2d_cpw_flip_chip_two_trace_zo_zm"
NOTEBOOK_RUN_DATE = date.today().isoformat()
NOTEBOOK_RUN_INDEX = 1
# NOTEBOOK_RUN_ID = f"{NOTEBOOK_RUN_DATE}-Run{NOTEBOOK_RUN_INDEX:02d}"
NOTEBOOK_RUN_ID = "2026-07-04-Run01"
AEDT_WORK_DIR = PATH.simulation / "aedt" / SIMULATION_PURPOSE_ID
AEDT_RUN_ROOT = AEDT_WORK_DIR / NOTEBOOK_RUN_ID
AEDT_PACKAGE_DIR = AEDT_RUN_ROOT
AEDT_ARCHIVE_PATH = AEDT_RUN_ROOT.parent / f"{AEDT_RUN_ROOT.name}-aedt.tar.gz"
AEDT_ANALYSIS_RUN_ROOT: Path | None = None
AEDT_PREPARE_RUN_STAGE = AEDT_ANALYSIS_RUN_ROOT is None
ACTIVE_AEDT_RUN_ROOT = Path(AEDT_ANALYSIS_RUN_ROOT or AEDT_RUN_ROOT)

# Manifest names that also define AEDT project/design result paths.
AEDT_PROJECT_NAME = "q2d_semantic_two_die_offset_sweep"
Q2D_TRACE_NAMES = ("T1", "T2")

AEDT_NUM_CORES = 4
AEDT_MAX_WORKERS = 7
AEDT_CORE_BUDGET = AEDT_NUM_CORES * AEDT_MAX_WORKERS
AEDT_MEMORY_MB_TOTAL = 240000

DIE_THICKNESS_UM = 500.0
AIR_HEIGHT_UM = 1000.0
GROUND_WIDTH_UM = 1000.0
METAL_THICKNESS_UM = 0.2

HORIZONTAL_OFFSET_UM = tuple(float(value) for value in range(0, 31, 3))
TRACE_GAP_UM = (3.0, 4.5, 6.0, 7.5, 9.0, 10.5)
FLIP_CHIP_GAP_HEIGHT_UM = tuple(value / 4 for value in range(22, 35))
CENTRAL_WIDTH_UM = tuple(float(value) for value in range(3, 8)) + tuple(
    float(value) for value in range(15, 26)
)

space = ParameterSpace(
    Axis("horizontal_offset_um", HORIZONTAL_OFFSET_UM, default=0.0),
    Axis("trace_gap_um", TRACE_GAP_UM, default=3.0),
    Axis("central_width_um", CENTRAL_WIDTH_UM, default=CENTRAL_WIDTH_UM[0]),
    Axis("flip_chip_gap_height_um", FLIP_CHIP_GAP_HEIGHT_UM, default=FLIP_CHIP_GAP_HEIGHT_UM[0]),
)


def aedt_point_slug(point) -> str:
    return point.id.replace("=", "_")


nominal_point = space.point()
offset_line = space.line("horizontal_offset_um", trace_gap_um=3.0, central_width_um=3.0)
width_line = space.line("central_width_um", horizontal_offset_um=0.0, trace_gap_um=3.0)
flip_gap_line = space.line(
    "flip_chip_gap_height_um",
    horizontal_offset_um=0.0,
    trace_gap_um=3.0,
    central_width_um=3.0,
)

(
    nominal_point,
    len(space.grid()),
    [aedt_point_slug(point) for point in offset_line[:3]],
    [point.coords["central_width_um"] for point in width_line],
    [point.coords["flip_chip_gap_height_um"] for point in flip_gap_line],
)

In [ ]:
def make_cross_section(
    *,
    horizontal_offset_um: float,
    trace_gap_um: float,
    central_width_um: float,
    flip_chip_gap_height_um: float,
) -> Q2dSemanticCrossSection:
    trace_center_local_um = GROUND_WIDTH_UM + trace_gap_um + central_width_um / 2.0
    d0_x0_um = -trace_center_local_um
    d1_x0_um = horizontal_offset_um - trace_center_local_um

    def sws_segments(name: str) -> tuple[Ground | Gap | Trace, ...]:
        return (
            Ground(width_um=GROUND_WIDTH_UM),
            Gap(width_um=trace_gap_um),
            Trace(name, width_um=central_width_um),
            Gap(width_um=trace_gap_um),
            Ground(width_um=GROUND_WIDTH_UM),
        )

    return Q2dSemanticCrossSection(
        stack=Stack(
            elements=(
                Air(height_um=AIR_HEIGHT_UM),
                Die(id="D0", thickness_um=DIE_THICKNESS_UM, material="Silicon"),
                DieGap(height_um=flip_chip_gap_height_um),
                Die(id="D1", thickness_um=DIE_THICKNESS_UM, material="Silicon"),
                Air(height_um=AIR_HEIGHT_UM),
            )
        ),
        face_patterns=(
            FacePattern(
                die="D0",
                face="top",
                metal_thickness_um=METAL_THICKNESS_UM,
                x0_um=d0_x0_um,
                segments=sws_segments("T1"),
            ),
            FacePattern(
                die="D1",
                face="bottom",
                metal_thickness_um=METAL_THICKNESS_UM,
                x0_um=d1_x0_um,
                segments=sws_segments("T2"),
            ),
        ),
    )

In [ ]:
def build_sweep_points(
    parameter_space: ParameterSpace,
    sidecar_dir: Path,
) -> list[dict[str, object]]:
    sidecar_dir.mkdir(parents=True, exist_ok=True)
    points = []
    for point in parameter_space.grid():
        point_slug = aedt_point_slug(point)
        cross_section_path = write_q2d_cross_section_payload(
            sidecar_dir / f"{point_slug}.json",
            make_cross_section(**point.coords),
        )
        points.append(
            {
                "point_slug": point_slug,
                "run_id": point_slug,
                "parameter_id": point.id,
                **{f"parameter_{key}": value for key, value in point.coords.items()},
                "q2d_cross_section_json_path": str(cross_section_path),
            }
        )
    return points


if AEDT_PREPARE_RUN_STAGE:
    cross_section_tempdir = TemporaryDirectory()
    sweep_points = build_sweep_points(space, Path(cross_section_tempdir.name))
    expected_point_count = 1
    for axis in space.axes:
        expected_point_count *= len(axis.values)
    assert len(sweep_points) == expected_point_count
    sweep_summary = {
        "points": len(sweep_points),
        "first_points": sweep_points[:3],
    }
else:
    cross_section_tempdir = None
    sweep_points = []
    sweep_summary = {
        "status": "analysis_only",
        "run_folder": str(ACTIVE_AEDT_RUN_ROOT),
    }

sweep_summary

In [ ]:
q2d_recipe = AedtRecipeSpec(
    id="q2d",
    type="q2d_extraction",
    q2d_geometry_mode="semantic_cross_section",
)


def write_points_metadata(
    package,
    points: list[dict[str, object]],
    parameter_space: ParameterSpace,
) -> None:
    point_rows = [
        {
            key: value
            for key, value in point.items()
            if key in {"point_slug", "run_id", "parameter_id"} or key.startswith("parameter_")
        }
        for point in points
    ]
    point_fieldnames = [
        "point_slug",
        "run_id",
        "parameter_id",
        *[f"parameter_{name}" for name in parameter_space.axis_names],
    ]
    with (package.package_dir / "points.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=point_fieldnames)
        writer.writeheader()
        writer.writerows(point_rows)
    (package.package_dir / "points.json").write_text(
        json.dumps(
            {"schema_version": "aedt-q2d-sweep-points.v1", "points": point_rows},
            indent=2,
        ),
        encoding="utf-8",
    )


if AEDT_PREPARE_RUN_STAGE:
    cases = tuple(
        AedtNativeCaseSpec(
            id=point["point_slug"],
            q2d_cross_section_json_path=Path(point["q2d_cross_section_json_path"]),
            recipes=(q2d_recipe,),
        )
        for point in sweep_points
    )
    package = prepare_aedt_native_handoff_package(
        AedtNativePackageSpec(
            project_name=AEDT_PROJECT_NAME,
            project_path=Path(f"{AEDT_PROJECT_NAME}.aedt"),
            point_local_sweep=True,
            hpc_resource=AedtHpcResourceSpec(
                num_cores=AEDT_NUM_CORES,
                max_workers=AEDT_MAX_WORKERS,
                core_budget=AEDT_CORE_BUDGET,
                memory_mb_total=AEDT_MEMORY_MB_TOTAL,
            ),
            cases=cases,
        ),
        package_dir=AEDT_PACKAGE_DIR,
        overwrite=True,
    )
    write_points_metadata(package, sweep_points, space)
    package_summary = {
        "cases": package.case_count,
        "manifest": str(package.manifest_path),
    }
else:
    package = None
    package_summary = {
        "status": "analysis_only",
        "manifest": str(ACTIVE_AEDT_RUN_ROOT / "manifest.yaml"),
    }

package_summary

In [ ]:
if AEDT_PREPARE_RUN_STAGE:
    archive = package_aedt_native_handoff(package, archive_path=AEDT_ARCHIVE_PATH)

    required_package_files = (
        package.manifest_path,
        package.import_run_config_path,
        package.solve_run_config_path,
        package.bash_script_path,
        package.scripts_dir / "runtime_bundle" / "run_aedt_native.py",
        package.package_dir / "points.csv",
        package.package_dir / "points.json",
        archive.archive_path,
    )
    missing_package_files = [str(path) for path in required_package_files if not path.is_file()]
    assert not missing_package_files, missing_package_files
    archive_summary = {
        "package_dir": str(package.package_dir.resolve()),
        "archive": str(archive.archive_path),
        "cases": package.case_count,
    }
else:
    archive = None
    archive_summary = {
        "status": "analysis_only",
        "package_dir": str(ACTIVE_AEDT_RUN_ROOT.resolve()),
    }

archive_summary

## Terminal Run Commands

Run these on the AEDT target machine after preparing the package-local PyAEDT environment from `README.md`. These commands target the single package generated from the current `space`.


In [ ]:
AEDT_RUN_ROOT_ABS = ACTIVE_AEDT_RUN_ROOT.resolve()
# The generated run_configs/*.yaml set skip_completed=True for point-local Q2D sweeps.
IMPORT_COMMAND = f"cd {AEDT_RUN_ROOT_ABS} && ./scripts/run_aedt_native.sh --mode import"
SOLVE_COMMAND = f"cd {AEDT_RUN_ROOT_ABS} && ./scripts/run_aedt_native.sh --mode solve"

print(IMPORT_COMMAND)
print(SOLVE_COMMAND)

## Layout-Backed Sweep Variant

The same `ParameterSpace` can drive a layout-backed sweep by passing `point.coords` into a GDSFactory component builder, then packaging those point-local GDS/TECH artifacts through the layout-backed AEDT package path.


In [ ]:
def layout_backed_component_inputs() -> list[dict[str, object]]:
    return [{"point_slug": aedt_point_slug(point), **point.coords} for point in space.grid()]


layout_backed_component_inputs()[:3]

## Analysis After AEDT Solve

This section is intentionally data-dependent. Before `--mode solve` finishes it reports missing matrix exports; after solve it loads raw AEDT matrix entries, derives explicit Zo/Zm metrics, and writes analysis artifacts under the run folder.


In [ ]:
raw_q2d = load_q2d_raw_sweep_result(
    ACTIVE_AEDT_RUN_ROOT,
    parameter_space=space,
    recipe_id=q2d_recipe.id,
)

derived_q2d = raw_q2d.derive(
    Q2dImpedanceFormula.self(name="zo", trace_names=Q2D_TRACE_NAMES),
    Q2dImpedanceFormula.mutual(name="zm", trace_pair=("T1", "T2")),
)

In [ ]:
(
    raw_q2d.write_csv(),
    derived_q2d.write_csv(),
    derived_q2d.write_formula_manifest(),
)

In [ ]:
width = derived_q2d.line(
    "central_width_um",
    horizontal_offset_um=0.0,
    trace_gap_um=3.0,
    flip_chip_gap_height_um=5.5,
)
width.write_csv()
width.rows  # noqa: B018
width.show_all_results()

In [ ]:
sweep_view = derived_q2d.slice(
    (
        "horizontal_offset_um",
        "trace_gap_um",
        "central_width_um",
        "flip_chip_gap_height_um",
    )
).where(
    horizontal_offset_um=lambda value: 0.0 <= value <= 20.0,
    central_width_um=lambda value: 7.0 <= value <= 20.0,
    flip_chip_gap_height_um=(7.0, 7.5, 8.0),
)

sweep_view.show(
    Q2dFacetLineGrid(
        x="horizontal_offset_um",
        y=(
            ("zo_T1_ohm", "Zo Trace1 (ohm)"),
            ("zo_T2_ohm", "Zo Trace2 (ohm)"),
            ("zm_T1_T2_ohm", "Zm Trace1-Trace2 (ohm)"),
        ),
        facet_col="trace_gap_um",
        color="central_width_um",
        line_dash="flip_chip_gap_height_um",
        line_dash_map={7.0: "solid", 7.5: "dash", 8.0: "dot"},
        title="Q2D Native Sweep: Zo and Zm vs Horizontal Offset",
        x_title="Horizontal offset (um)",
        color_title="Central metal width (um)",
        facet_col_title="CPW gap = {value:g} um",
        shared_y=True,
    )
)